# Use case #1 — Debug an error with an LLM
## Semantic Search (embeddings)

**Same incident. Same operator. This time they didn't paste the log line.**

## 1. Embed the same 71 sections

In [1]:
import hashlib, json
from pathlib import Path

import numpy as np
from openai import OpenAI

from makani import load_chunks, tokenize, bm25_index, show, api_key

SPLIT = "sections"   # "sections" = split on ## headers | "fixed" = 800-char chunks, no structure

chunks = load_chunks(split=SPLIT)
bm25 = bm25_index(chunks)
oai = OpenAI(api_key=api_key("OPENAI_KEY"))
CACHE = Path("embeddings.json")


def embed(texts):
    """Embed, reusing anything we've embedded before (keyed by text hash)."""
    cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}
    keys = [hashlib.sha1(t.encode()).hexdigest()[:16] for t in texts]
    todo = [t for t, k in zip(texts, keys) if k not in cache]
    if todo:
        got = oai.embeddings.create(model="text-embedding-3-small", input=todo).data
        cache.update({hashlib.sha1(t.encode()).hexdigest()[:16]: e.embedding
                      for t, e in zip(todo, got)})
        CACHE.write_text(json.dumps(cache))
    return np.array([cache[k] for k in keys])


def unit(v):
    """Scale to length 1, so cosine similarity is just a dot product."""
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


E = unit(embed([c.text for c in chunks]))   # one row per chunk: (n_chunks, 1536)
print(f"{len(chunks)} chunks embedded ({SPLIT}), {E.shape[1]} dimensions each -> cached in {CACHE}")


# `@` is matrix multiply. E is (71, 1536), the query vector is (1536,), so E @ q pairs
# each row with the query and returns (71,) -- one score per chunk. It is the same as
# [np.dot(row, q) for row in E], about 50x faster.
#
# Cosine similarity is dot(a, b) / (|a| * |b|). Both vectors are already unit length
# from unit(), so both magnitudes are 1, the division disappears, and cosine *is* the
# dot product. That's why we normalise once up front instead of per query.
#
# The whole semantic search is one BLAS call: 71 x 1536 multiply-adds, microseconds.
# You need a vector database at a million chunks, not at seventy-one.
def semantic(query):
    return E @ unit(embed([query])[0])

71 chunks embedded (sections), 1536 dimensions each -> cached in embeddings.json


## 2. The question BM25 cannot answer

In [2]:
query = ("Makani won't start after last night's deploy and the logs mention "
         "something it needed but could not find. What should I do?")

print("BM25\n")
show(chunks, bm25.get_scores(tokenize(query)), top_k=5)
print("\nEMBEDDINGS\n")
show(chunks, semantic(query), top_k=5, fmt="{:6.3f}")

BM25

  11.835   kesh-auth-errors.md             § Kesh won't start
  10.833   sapir-config-errors-legacy.md   § CONFIGURATION_IS_MISSING  ⚠️ superseded
   9.920   onboarding-makani.md            § How Makani is organized
   9.215   sapir-config-errors.md          § Sapir — Configuration Errors
   8.961   nugat-write-path-errors.md      § Nugat won't start

EMBEDDINGS

   0.565   onboarding-makani.md            § How Makani is organized
   0.543   sapir-config-errors.md          § CONFIGURATION_IS_MISSING
   0.511   makani-architecture-overview.md § Makani — Architecture Overview
   0.501   incident-2026-03-config-outage. § Summary
   0.492   sapir-config-errors-legacy.md   § CONFIGURATION_IS_MISSING  ⚠️ superseded


**BM25 didn't just miss — it found the stale twin.** Its #2 is `sapir-config-errors-legacy.md`, the 2024 page. The **current** version of that exact section is ranked **#16**, far outside any context window you'd send to an LLM. Without the rare literal string, all BM25 has left is `start` / `logs` / `find`, and those words point everywhere: Kesh, the onboarding doc, Vello.

**Embeddings put the correct current section at #2.** Not #1 — the onboarding doc's *"always check Sapir's logs first"* is genuinely a good semantic match for "won't start". But at any realistic `k`, semantic retrieval puts the answer in the prompt and lexical does not.

That distinction is invisible in the score list and decisive in the output below — same model, same question, top-3 sections from each ranking.

In [3]:
# what the LLM actually does with each retrieval
from makani import render   # provenance header (component / owner / status / last_updated) + body

llm = OpenAI(api_key=api_key("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")


def answer(query, scores, k=3, extra=""):
    top = [chunks[i] for i in sorted(range(len(chunks)), key=lambda i: -scores[i])[:k]]
    context = "\n\n".join(render(c) for c in top)   # <- the model DOES see the dates
    for c in top:
        print(f"   ctx: {c.filename} § {c.heading}   [{c.meta.get('last_updated')}]")
    # gpt-oss-120b is a reasoning model: reasoning tokens occasionally eat the whole
    # budget and content comes back empty. Retry rather than print None on stage.
    for _ in range(3):
        r = llm.chat.completions.create(
            model="openai/gpt-oss-120b", max_tokens=1000,
            messages=[{"role": "user", "content":
                       f"Answer the operator using ONLY this documentation.{extra}\n\n{context}\n\n"
                       f"Operator: {query}\nAnswer in 2-3 sentences."}])
        if r.choices[0].message.content:
            return print("\n", r.choices[0].message.content, "\n")


print("— top 3 from BM25 —")
answer(query, bm25.get_scores(tokenize(query)))
print("— top 3 from embeddings —")
answer(query, semantic(query))

— top 3 from BM25 —
   ctx: kesh-auth-errors.md § Kesh won't start   [2026-07-01]
   ctx: sapir-config-errors-legacy.md § CONFIGURATION_IS_MISSING   [2024-03-08]
   ctx: onboarding-makani.md § How Makani is organized   [2026-01-10]



 Check Sapir’s logs for a `CONFIGURATION_IS_MISSING` error – that means a required Makani key is absent from `sapir.conf`. Open `sapir.conf`, add the missing key in the `[required]` section, then do a full restart of Makani (the old `--reload-config` flag is gone). If the error persists after the restart, page the platform‑team for further help. 

— top 3 from embeddings —
   ctx: onboarding-makani.md § How Makani is organized   [2026-01-10]
   ctx: sapir-config-errors.md § CONFIGURATION_IS_MISSING   [2026-06-14]
   ctx: makani-architecture-overview.md § Makani — Architecture Overview   [2026-06-30]



 Check Sapir’s logs first – the line right before the failure will name the missing key (Sapir prints it just above a `CONFIGURATION_IS_MISSING` error). Verify that this key is defined in `makani.config.yaml` for the current environment or set as an appropriate environment variable, then restart Sapir so it can re‑validate the configuration. If the key is present and the error persists, repeat the check after the restart. 



## 3. Where embeddings lose

Two failures, and the second is the same wall we hit in notebook `01`.

In [4]:
q1 = "We saw the following Makani error in the logs: CONFIGURATION_IS_MISSING, what should I do?"

print("BM25\n")
show(chunks, bm25.get_scores(tokenize(q1)), top_k=4)
print("\nEMBEDDINGS\n")
show(chunks, semantic(q1), top_k=4, fmt="{:6.3f}")

BM25

  17.419   sapir-config-errors-legacy.md   § CONFIGURATION_IS_MISSING  ⚠️ superseded
  14.194   sapir-config-errors.md          § CONFIGURATION_IS_MISSING
  12.786   runbook-startup-failures.md     § If Sapir is the failed component
  10.316   sapir-conf-migration-2025.md    § Errors during migration  ⚠️ superseded

EMBEDDINGS

   0.719   sapir-config-errors-legacy.md   § CONFIGURATION_IS_MISSING  ⚠️ superseded
   0.691   sapir-config-errors.md          § CONFIGURATION_IS_MISSING
   0.584   makani-config-reference.md      § Makani — Configuration Key Reference
   0.584   sapir-config-errors.md          § Quick reference


**"Just tell the model to prefer the newest doc."** Reasonable objection — the dates are right there in the context. Let's try it.

In [5]:
RECENCY = ("\n\nIMPORTANT: each source is stamped with `last updated: YYYY-MM-DD` and a "
           "`status`. Prefer the most recently updated source. If a source is marked "
           "superseded, do not follow its instructions; say so explicitly instead.")

print("— same top 3 from BM25, now WITH an explicit recency instruction —")
answer(query, bm25.get_scores(tokenize(query)), extra=RECENCY)

— same top 3 from BM25, now WITH an explicit recency instruction —
   ctx: kesh-auth-errors.md § Kesh won't start   [2026-07-01]
   ctx: sapir-config-errors-legacy.md § CONFIGURATION_IS_MISSING   [2024-03-08]
   ctx: onboarding-makani.md § How Makani is organized   [2026-01-10]



 Check Sapir’s logs first — Makani boots only after Sapir validates its configuration, and most “Makani won’t start” cases are caused by a missing or bad Sapir config key. Fix the reported configuration problem in `sapir.conf` and then do a full restart of Makani. (The older “CONFIGURATION_IS_MISSING” guide in *sapir‑config‑errors‑legacy.md* is superseded and should not be used.) 



**It notices, and it still can't help you.** The model correctly flags the legacy page as superseded — and then hands over its instructions anyway, because they are the only instructions in the context. One answer, two contradictory halves.

That is the precise shape of the limitation: **a prompt can make the model *notice* the problem; it cannot make it *replace* the missing information.** The current procedure was ranked #16 and never retrieved, so no instruction can conjure it. Put both versions in context and the same prompt gets it right immediately — which means the fix belongs in retrieval, not in the prompt.

Arguably this is worse than the un-instructed answer: a confidently wrong response is at least easy to catch in review. A self-contradicting one at 3am is not.

**1. Put the log line back and embeddings stop helping.** Same top two as BM25, same wrong order — but look at the margins: BM25 spreads them 17.4 vs 14.2 (**23%**), the embedding 0.719 vs 0.691 (**4%**). Everything *about* configuration errors points nearly the same direction, so cosine similarity compresses the range exactly where you needed discrimination. The rare literal string was already a perfect signal, and embedding it throws that precision away.

So neither method dominates: **semantic wins when the operator describes, lexical wins when they paste.** You don't know which one you'll get. That's the argument for **hybrid** — run both, fuse the rankings — not for replacing `01`.

**2. The ranking is date-blind, and no prompt fixes that.** The superseded 2024 page is #1 in both rankings. Not because we hid the dates — `render()` puts `status` and `last_updated` in every context block, and the cell above shows that even an explicit *"prefer the newest, refuse superseded"* instruction only buys a warning stapled to the same stale advice. Nothing in the question asked about freshness, so nothing in the *score* reflects it; and at `k=3` the current section was never retrieved, so there was nothing to prefer.

`status` and `last_updated` are parsed, in `c.meta`, and in the prompt. Neither ranking function can touch them.

→ **`03_structured_search.ipynb`**: stop scoring, start filtering.